# DeiT-Tiny seed2 — projection-aware exploration: D0 / A3 / A4

**Input:** CIFAR-100 plus the original seed2 historical-best fork `.pt` or an
output archive containing it: **epoch375, validation accuracy56.20%, protocol v5**.
No Vanilla plateau training or old arm training runs here. Old `result.json` files
are optional for comparison; they must have the same fork hash, protocol, CP config
and horizon. Attach full original output archives for reliable hash preservation.
Enable Internet, T4x2 and Kaggle Secret `github_token`.

1. D0 scans all12 MLP sites, projects E and the negative summed-CE logit gradient
   into each same block, previews the gate grid, saves CSV/summary, and commits nothing.
2. A3 `e2o_top3_best_gate`: raw TINY rank → evaluate all top3 projections on
   the same untouched anchor/gate batch → commit at most one positive finite gate correction.
3. A4 `e2o_top3_recurrent`: same first search, plus one fresh top3 search after
   each rollback while training budget remains. No intervention after the final epoch.

A3 and A4 run independently from identical full fork state on two GPUs,150
post-fork training epochs each. Patience10, original rank4/scales(.0125,.025,.05),
WHERE3x32, projection32, gate32, adaptive dual-CG200, batch64, inherited AdamW/LR.
No cross-block projection; later training optimizes the full original network.
Gate is a selection batch, not unbiased held-out evidence. Validation never
selects site, damping, scale or a candidate; it only observes committed states,
updates report/anchor and triggers rollback.

Controller semantics are inherited: report best is strict accuracy with paired
loss; anchor is higher accuracy OR exact accuracy tie with lower loss, and stall
resets on anchor improvement. Rollback restores model/optimizer/scheduler,
preserves current RNG/loader stream. Recurrent searches draw fresh probe seeds
from current CPU RNG; the first search uses the original index0 partition.
A separate fixed original projection batch measures direction cosine across
searches and is never used to select candidates. No claim of a different direction
follows merely from selecting the same block or using different RNG.

Full latest checkpoints include committed events, prior direction and pending
search state. Resume skips complete arms and completes a pending rollback search
without repeating an already committed intervention. Outputs: D0 CSV+summary;
per arm result.json, interventions.jsonl, epoch_history.csv, best_checkpoint.pt,
run_metadata.json, checkpoint_latest.pt. Best observed after training and stored
anchor are reported separately. A3/A4 spend more proposal/CG compute than top1;
comparisons do not establish an equal-compute advantage.


In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO = Path("/kaggle/working/deit-all-arms-repo")
OUTPUT = Path("/kaggle/working/projection_log_analysis")
URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
token = UserSecretsClient().get_secret("github_token").strip()
if not token:
    raise RuntimeError("Enable the github_token Kaggle Secret")
askpass = Path("/kaggle/working/.projection_log_git_askpass.py")
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
try:
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", URL, str(REPO)], env=env, check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "main"], env=env, check=True)
        subprocess.run(["git", "-C", str(REPO), "merge", "--ff-only", "origin/main"], env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop("GITHUB_TOKEN_RUNTIME", None)
    del token
GROMO = Path("/kaggle/working/deit-gromo")
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
if not GROMO.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                    "https://github.com/growingnet/gromo.git", str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1", "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach", GROMO_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[vision,test]", "-e", str(GROMO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
ENV = os.environ.copy()
ENV["PYTHONPATH"] = os.pathsep.join((str(REPO), str(GROMO / "src")))
ENV["REQUIRE_DEIT_INTEGRATION"] = "1"
subprocess.run([sys.executable, "-m", "pytest", "-q",
    "tests/test_deit_function_preserving.py", "tests/test_deit_sites.py",
    "tests/test_deit_projection.py", "tests/test_deit_no_aux_persistence.py",
    "tests/test_deit_protocol.py", "tests/test_deit_vanilla_reference.py", "tests/test_deit_e_rollback.py", "tests/test_deit_ablation.py", "tests/test_deit_ablation_integration.py", "tests/test_deit_logging.py", "tests/test_deit_parallel.py", "tests/test_deit_resume.py", "tests/test_deit_projection_aware.py", "tests/test_kaggle_checkpoint_discovery.py"], cwd=REPO, env=ENV, check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable a CUDA GPU; this notebook trains DeiT")
print("Repository commit:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
import os
OUTPUT = Path('/kaggle/working/deit_seed2_projection_aware_v1')
OUTPUT.mkdir(parents=True, exist_ok=True)
GPU_DEVICES = 'auto'
roots = set()
for directory, _, files in os.walk('/kaggle/input', followlinks=True):
    if Path(directory).name == 'cifar-100-python' and {'train', 'meta'}.issubset(files):
        roots.add(Path(directory).parent.resolve())
if len(roots) != 1:
    raise FileNotFoundError(f'Attach exactly one CIFAR-100 dataset: {roots}')
DATA_ROOT = next(iter(roots))


In [ ]:
# CPU-only discovery; requires a completed fork, never trains Vanilla.
from experiments.deit_projection_resume import prepare_projection_resume
PLAN = prepare_projection_resume(['/kaggle/input'], OUTPUT)


In [ ]:
command = [sys.executable, '-m', 'experiments.run_deit_projection_aware_suite',
    '--data-root', str(DATA_ROOT), '--output', str(OUTPUT),
    '--resume-root', '/kaggle/input', '--gpu-devices', GPU_DEVICES]
try:
    subprocess.run(command, cwd=REPO, env=ENV, check=True)
finally:
    archive = shutil.make_archive(str(OUTPUT), 'gztar', root_dir=OUTPUT)
    print('Resumable output archive:', archive)


In [ ]:
for name in ('all12_transfer_diagnostic/result.json', 'comparison.json', 'arm_status.json'):
    path = OUTPUT / name
    if path.exists():
        result = json.loads(path.read_text())
        print(name, json.dumps(result.get('summary', result), indent=2))
